In [0]:
catalogs = spark.sql("SHOW CATALOGS")
catalogs.show(truncate=False)

+---------+
|catalog  |
+---------+
|samples  |
|system   |
|workspace|
+---------+



In [0]:
schemas = spark.sql("SHOW SCHEMAS IN workspace")
schemas.show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
+------------------+



In [0]:
volumes = spark.sql("SHOW VOLUMES IN workspace.default")
volumes.show(truncate=False)

+--------+----------------+
|database|volume_name     |
+--------+----------------+
|default |clickstream_data|
+--------+----------------+



In [0]:
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.default.clickstream_data
""")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5833099451259994>, line 1
----> 1 spark.sql("""
      2     CREATE VOLUME workspace.default.clickstream_data
      3 """)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:919, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    916         _views.append(SubqueryAlias(df._plan, name))
    918 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 919 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    920 if "sql_command_result" in properties:
    921     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1812, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1810 req = self._execute_plan_request_with_metad

In [0]:
spark.sql(
    "SHOW VOLUMES IN workspace.default"
).show(truncate=False)

+--------+----------------+
|database|volume_name     |
+--------+----------------+
|default |clickstream_data|
+--------+----------------+



In [0]:
input_path = "/Volumes/workspace/default/clickstream_data/e-shop clothing 2008.csv"

clicks = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", True)
    .csv(input_path)
)

clicks.printSchema()
print("Row count:", clicks.count())
display(clicks.limit(5))

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session ID: integer (nullable = true)
 |-- page 1 (main category): integer (nullable = true)
 |-- page 2 (clothing model): string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- model photography: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price 2: integer (nullable = true)
 |-- page: integer (nullable = true)

Row count: 165474


year,month,day,order,country,session ID,page 1 (main category),page 2 (clothing model),colour,location,model photography,price,price 2,page
2008,4,1,1,29,1,1,A13,1,5,1,28,2,1
2008,4,1,2,29,1,1,A16,1,6,1,33,2,1
2008,4,1,3,29,1,2,B4,10,2,1,52,1,1
2008,4,1,4,29,1,2,B17,6,6,2,38,2,1
2008,4,1,5,29,1,2,B8,4,3,2,52,1,1


In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
)

schema = StructType([
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day", IntegerType(), True),
    StructField("order", IntegerType(), True),
    StructField("country", IntegerType(), True),
    StructField("session ID", IntegerType(), True),
    StructField("page 1 (main category)", IntegerType(), True),
    StructField("page 2 (clothing model)", StringType(), True),
    StructField("colour", IntegerType(), True),
    StructField("location", IntegerType(), True),
    StructField("model photography", IntegerType(), True),
    StructField("price", IntegerType(), True),
    StructField("price 2", IntegerType(), True),
    StructField("page", IntegerType(), True),
])

raw_clicks = (
    spark.read
    .schema(schema)
    .option("header", True)
    .option("sep", ";")
    .option("mode", "FAILFAST")
    .option("enforceSchema", False)
    .csv(input_path)
)

raw_clicks.printSchema()
print("Row count:", raw_clicks.count())

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session ID: integer (nullable = true)
 |-- page 1 (main category): integer (nullable = true)
 |-- page 2 (clothing model): string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- model photography: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price 2: integer (nullable = true)
 |-- page: integer (nullable = true)

Row count: 165474


In [0]:
from pyspark.sql import functions as F

silver_preview = (
    raw_clicks
    .withColumnRenamed("order", "click_order")
    .withColumnRenamed("session ID", "session_id")
    .withColumnRenamed("page 1 (main category)", "main_category")
    .withColumnRenamed("page 2 (clothing model)", "product_code")
    .withColumnRenamed("colour", "color")
    .withColumnRenamed("model photography", "model_photography")
    .withColumnRenamed("price 2", "price_above_category_avg")
    .withColumn(
        "event_date",
        F.make_date(
            F.col("year"),
            F.col("month"),
            F.col("day")
        )
    )
)

print("Rows:", silver_preview.count())
print("Missing event dates:", silver_preview.filter(F.col("event_date").isNull()).count())

display(
    silver_preview.select(
        "event_date",
        "session_id",
        "click_order",
        "main_category",
        "product_code",
        "price"
    ).limit(5)
)

Rows: 165474
Missing event dates: 0


event_date,session_id,click_order,main_category,product_code,price
2008-04-01,1,1,1,A13,28
2008-04-01,1,2,1,A16,33
2008-04-01,1,3,2,B4,52
2008-04-01,1,4,2,B17,38
2008-04-01,1,5,2,B8,52


In [0]:
# Count missing values in each column
missing_counts = silver_preview.select(
    *[
        F.count(
            F.when(F.col(column).isNull(), 1)
        ).alias(column.replace(" ", "_"))
        for column in silver_preview.columns
    ]
)

display(missing_counts)

year,month,day,click_order,country,session_id,main_category,product_code,color,location,model_photography,price,price_above_category_avg,page,event_date
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Count clicks that share the same session and order
duplicate_clicks = (
    silver_preview
    .groupBy("session_id", "click_order")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate session-order pairs:", duplicate_clicks.count())

Duplicate session-order pairs: 0


In [0]:
# Check the date range and number of distinct dates
silver_preview.select(
    F.min("event_date").alias("first_date"),
    F.max("event_date").alias("last_date"),
    F.countDistinct("event_date").alias("distinct_dates")
).show()

+----------+----------+--------------+
|first_date| last_date|distinct_dates|
+----------+----------+--------------+
|2008-04-01|2008-08-13|           135|
+----------+----------+--------------+



In [0]:
# Save the cleaned clickstream data as a Delta table
table_name = "workspace.default.clickstream_silver"

(
    silver_preview.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

print("Saved table:", table_name)

Saved table: workspace.default.clickstream_silver


In [0]:
# Read the saved Delta table and verify its contents
saved_clicks = spark.table("workspace.default.clickstream_silver")

saved_clicks.select(
    F.count("*").alias("row_count"),
    F.min("event_date").alias("first_date"),
    F.max("event_date").alias("last_date"),
    F.countDistinct("event_date").alias("distinct_dates")
).show()

+---------+----------+----------+--------------+
|row_count|first_date| last_date|distinct_dates|
+---------+----------+----------+--------------+
|   165474|2008-04-01|2008-08-13|           135|
+---------+----------+----------+--------------+



In [0]:
# Summarize click activity for each session
session_summary = (
    saved_clicks
    .groupBy("session_id", "event_date")
    .agg(
        F.count("*").alias("click_count"),
        F.countDistinct("product_code").alias("distinct_product_count"),
        F.countDistinct("main_category").alias("distinct_category_count"),
        F.min("click_order").alias("first_click_order"),
        F.max("click_order").alias("last_click_order")
    )
)

print("Session summary rows:", session_summary.count())
display(session_summary.orderBy("session_id").limit(5))

Session summary rows: 24026


session_id,event_date,click_count,distinct_product_count,distinct_category_count,first_click_order,last_click_order
1,2008-04-01,9,9,4,1,9
2,2008-04-01,10,8,3,1,10
3,2008-04-01,6,6,3,1,6
4,2008-04-01,4,4,2,1,4
5,2008-04-01,1,1,1,1,1


In [0]:
# Summarize clicks and sessions by date
daily_summary = (
    session_summary
    .groupBy("event_date")
    .agg(
        F.sum("click_count").alias("daily_clicks"),
        F.countDistinct("session_id").alias("daily_sessions")
    )
)

# Check the number of dates and overall totals
daily_summary.agg(
    F.count("*").alias("days"),
    F.sum("daily_clicks").alias("total_clicks"),
    F.sum("daily_sessions").alias("total_sessions")
).show()

display(daily_summary.orderBy("event_date").limit(5))

+----+------------+--------------+
|days|total_clicks|total_sessions|
+----+------------+--------------+
| 135|      165474|         24026|
+----+------------+--------------+



event_date,daily_clicks,daily_sessions
2008-04-01,3181,477
2008-04-02,3456,480
2008-04-03,1854,276
2008-04-04,1801,266
2008-04-05,1504,199


In [0]:
# Save the session-level summary
session_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.clickstream_sessions")

# Save the daily summary
daily_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.clickstream_daily")

print("Saved tables:")
print("- workspace.default.clickstream_sessions")
print("- workspace.default.clickstream_daily")

Saved tables:
- workspace.default.clickstream_sessions
- workspace.default.clickstream_daily


In [0]:
# Read both saved summary tables
saved_sessions = spark.table("workspace.default.clickstream_sessions")
saved_daily = spark.table("workspace.default.clickstream_daily")

print("Session table rows:", saved_sessions.count())
print("Daily table rows:", saved_daily.count())

Session table rows: 24026
Daily table rows: 135


In [0]:
# Summarize the distribution of clicks per session
saved_sessions.select("click_count").summary(
    "count", "mean", "50%", "min", "max"
).show()

# Count sessions in click-count ranges
saved_sessions.withColumn(
    "click_range",
    F.when(F.col("click_count") == 1, "1 click")
     .when(F.col("click_count").between(2, 5), "2-5 clicks")
     .when(F.col("click_count").between(6, 10), "6-10 clicks")
     .otherwise("11+ clicks")
).groupBy("click_range").count().orderBy("click_range").show()

+-------+------------------+
|summary|       click_count|
+-------+------------------+
|  count|             24026|
|   mean|6.8872887704986265|
|    50%|                 4|
|    min|                 1|
|    max|               195|
+-------+------------------+

+-----------+-----+
|click_range|count|
+-----------+-----+
|    1 click| 5042|
| 11+ clicks| 4564|
| 2-5 clicks| 9574|
|6-10 clicks| 4846|
+-----------+-----+



In [0]:
# Summarize clicks and average product price by category
category_summary = (
    saved_clicks
    .groupBy("main_category")
    .agg(
        F.count("*").alias("clicks"),
        F.countDistinct("session_id").alias("sessions"),
        F.round(F.avg("price"), 2).alias("avg_price")
    )
    .orderBy("main_category")
)

display(category_summary)

main_category,clicks,sessions,avg_price
1,49742,13082,46.71
2,38408,11490,51.19
3,38577,10724,40.29
4,38747,9192,36.23


In [0]:
# Add readable category names
category_summary_named = category_summary.withColumn(
    "category_name",
    F.when(F.col("main_category") == 1, "Trousers")
     .when(F.col("main_category") == 2, "Skirts")
     .when(F.col("main_category") == 3, "Blouses")
     .when(F.col("main_category") == 4, "Sale")
     .otherwise("Unknown")
).select(
    "category_name",
    "clicks",
    "sessions",
    "avg_price"
)

display(category_summary_named)

category_name,clicks,sessions,avg_price
Trousers,49742,13082,46.71
Skirts,38408,11490,51.19
Blouses,38577,10724,40.29
Sale,38747,9192,36.23


In [0]:
# Find the dates with the highest and lowest click counts
print("Top 10 days by clicks:")
display(
    saved_daily
    .orderBy(F.col("daily_clicks").desc())
    .limit(10)
)

print("Bottom 10 days by clicks:")
display(
    saved_daily
    .orderBy(F.col("daily_clicks").asc())
    .limit(10)
)

Top 10 days by clicks:


event_date,daily_clicks,daily_sessions
2008-04-02,3456,480
2008-04-01,3181,477
2008-04-10,1854,258
2008-04-03,1854,276
2008-04-18,1833,221
2008-04-07,1801,276
2008-04-04,1801,266
2008-04-29,1777,230
2008-04-14,1754,237
2008-04-08,1717,234


Bottom 10 days by clicks:


event_date,daily_clicks,daily_sessions
2008-08-13,200,21
2008-07-19,538,102
2008-05-03,594,108
2008-05-31,621,96
2008-06-14,627,96
2008-06-21,645,95
2008-08-02,714,117
2008-06-28,749,123
2008-05-17,767,113
2008-06-15,781,124


In [0]:
# Measure the correlation between daily clicks and daily sessions
daily_correlation = saved_daily.stat.corr(
    "daily_clicks",
    "daily_sessions"
)

print("Correlation between daily clicks and sessions:", daily_correlation)

Correlation between daily clicks and sessions: 0.9480802328527075


In [0]:
# Calculate the average number of clicks per session for each day
daily_engagement = (
    saved_daily
    .withColumn(
        "clicks_per_session",
        F.round(
            F.col("daily_clicks") / F.col("daily_sessions"),
            2
        )
    )
    .orderBy(F.col("clicks_per_session").desc())
)

display(daily_engagement.limit(10))

event_date,daily_clicks,daily_sessions,clicks_per_session
2008-08-13,200,21,9.52
2008-08-06,1651,175,9.43
2008-07-26,898,100,8.98
2008-05-08,1409,161,8.75
2008-04-19,1537,180,8.54
2008-06-22,862,103,8.37
2008-04-18,1833,221,8.29
2008-08-08,1267,153,8.28
2008-08-03,1022,124,8.24
2008-05-25,1425,173,8.24


In [0]:
# Summarize daily clicks per session across all dates
daily_engagement.select("clicks_per_session").summary(
    "count", "mean", "50%", "min", "max"
).show()

# Find the top 10 days with at least 100 sessions
display(
    daily_engagement
    .filter(F.col("daily_sessions") >= 100)
    .orderBy(F.col("clicks_per_session").desc())
    .limit(10)
)

+-------+------------------+
|summary|clicks_per_session|
+-------+------------------+
|  count|               135|
|   mean| 6.914814814814812|
|    50%|              6.81|
|    min|              5.27|
|    max|              9.52|
+-------+------------------+



event_date,daily_clicks,daily_sessions,clicks_per_session
2008-08-06,1651,175,9.43
2008-07-26,898,100,8.98
2008-05-08,1409,161,8.75
2008-04-19,1537,180,8.54
2008-06-22,862,103,8.37
2008-04-18,1833,221,8.29
2008-08-08,1267,153,8.28
2008-08-03,1022,124,8.24
2008-05-25,1425,173,8.24
2008-05-29,1494,185,8.08


In [0]:
# Measure the relationship between session length and distinct products viewed
session_correlation = saved_sessions.stat.corr(
    "click_count",
    "distinct_product_count"
)

print(
    "Correlation between clicks and distinct products:",
    session_correlation
)

# Compare average distinct products across session-length groups
session_depth = (
    saved_sessions
    .withColumn(
        "click_range",
        F.when(F.col("click_count") == 1, "1 click")
         .when(F.col("click_count").between(2, 5), "2-5 clicks")
         .when(F.col("click_count").between(6, 10), "6-10 clicks")
         .otherwise("11+ clicks")
    )
    .groupBy("click_range")
    .agg(
        F.count("*").alias("sessions"),
        F.round(F.avg("distinct_product_count"), 2)
         .alias("avg_distinct_products")
    )
)

display(session_depth)

Correlation between clicks and distinct products: 0.9713972619693167


click_range,sessions,avg_distinct_products
1 click,5042,1.0
2-5 clicks,9574,3.02
11+ clicks,4564,17.58
6-10 clicks,4846,7.06


In [0]:
# Calculate repeated product views within each session
session_repeats = saved_sessions.withColumn(
    "repeat_product_views",
    F.col("click_count") - F.col("distinct_product_count")
)

print(
    "Sessions with at least one repeat product view:",
    session_repeats
    .filter(F.col("repeat_product_views") > 0)
    .count()
)

session_repeats.select(
    F.round(F.avg("repeat_product_views"), 2)
     .alias("avg_repeat_views_per_session"),
    F.max("repeat_product_views")
     .alias("max_repeat_views")
).show()

display(
    session_repeats
    .orderBy(F.col("repeat_product_views").desc())
    .select(
        "session_id",
        "event_date",
        "click_count",
        "distinct_product_count",
        "repeat_product_views"
    )
    .limit(10)
)

Sessions with at least one repeat product view: 6046
+----------------------------+----------------+
|avg_repeat_views_per_session|max_repeat_views|
+----------------------------+----------------+
|                        0.71|              60|
+----------------------------+----------------+



session_id,event_date,click_count,distinct_product_count,repeat_product_views
8790,2008-05-12,121,61,60
13846,2008-06-10,192,134,58
23383,2008-08-09,108,51,57
14336,2008-06-13,95,39,56
6592,2008-04-29,112,60,52
2363,2008-04-08,106,56,50
9309,2008-05-15,104,57,47
2783,2008-04-10,88,42,46
3515,2008-04-14,73,31,42
17209,2008-07-02,101,60,41


## Project Findings

### Data processing
- The source data contains 165,474 click records from 24,026 sessions across 135 dates, from 2008-04-01 to 2008-08-13.
- The cleaned click data and the session-level and daily summary tables were saved as Delta tables.
- Row counts and date totals matched the source data after processing.

### Browsing behavior
- A session had an average of 6.89 clicks; the median was 4 clicks.
- The longest session had 195 clicks.
- Trousers received the most clicks (49,742). Skirts had the highest average listed price ($51.19), while Sale had the lowest ($36.23).
- Daily clicks and sessions were strongly correlated (r = 0.949).
- 6,046 sessions included at least one repeat product view.

### Interpretation and limitations
- These results describe browsing activity, not purchases or sales. The dataset has no purchase or conversion outcome.
- A repeat product view means the same product code appeared more than once in a session; it does not explain why the user returned to that product.
- Daily clicks per session can be unstable on dates with few sessions.


In [0]:
# Check the saved cleaned table schema and row count
saved_clicks = spark.table("workspace.default.clickstream_silver")

saved_clicks.printSchema()
print("Saved click rows:", saved_clicks.count())

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- click_order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- main_category: integer (nullable = true)
 |-- product_code: string (nullable = true)
 |-- color: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- model_photography: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price_above_category_avg: integer (nullable = true)
 |-- page: integer (nullable = true)
 |-- event_date: date (nullable = true)

Saved click rows: 165474
